<a href="https://colab.research.google.com/github/raddified/Vehicle_Damage_Severity_Classification/blob/main/Model_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torchvision
import albumentations
import wandb

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("issamjebnouni/cardd")

print("Path to dataset files:", path)

100%|██████████| 2.81G/2.81G [00:37<00:00, 79.5MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/issamjebnouni/cardd/versions/1


In [ ]:
def map_to_ordinal_rank(label_name: str, bbox: list, img_width: int, img_height: int) -> int:
    """
    Maps categorical CarDD labels to Ordinal Severity Ranks (1=Minor, 2=Moderate, 3=Severe).
    bbox format assumed: [x_min, y_min, x_max, y_max]
    """
    label = label_name.lower()

    # Rank 1: Minor (Surface level)
    if label in ['scratch', 'crack']:
        return 1

    # Rank 3: Severe (Safety/Structural)
    if label in ['glass shatter', 'tire flat']:
        return 3

    # Rank 2 or 3: Dents and Broken Lamps
    if label in ['dent', 'lamp broken']:
        # Calculate bounding box area
        bbox_width = bbox[2] - bbox[0]
        bbox_height = bbox[3] - bbox[1]
        bbox_area = bbox_width * bbox_height
        image_area = img_width * img_height

        # If a dent covers more than 30% of the image, upgrade to Severe
        if (bbox_area / image_area) > 0.30:
            return 3
        return 2

    # Fallback for unknown classes
    return 1